# PhasePhyto Apple Overlap Colab Pipeline

This notebook downloads the three apple-overlap source datasets into Google Drive, stores them as **tar archives** to save memory and speed up future sessions, prepares the strict shared-label overlap subset, archives that overlap subset too, then hydrates the overlap archive onto Colab SSD for training/evaluation.

Shared labels:

- `Apple___healthy`
- `Apple___Apple_scab`
- `Apple___Cedar_apple_rust`

Datasets used:

- PlantVillage
- PlantDoc
- Plant Pathology 2021


In [ ]:
# ============================================================
# CONFIGURATION -- edit only if you want different paths/behavior
# ============================================================

CONFIG = {
    "drive_project_dir": "/content/drive/MyDrive/PhasePhyto",
    "repo_url": "https://github.com/kautilyaa/PhasePhyto.git",
    "repo_branch": "main",
    "repo_dir": "/content/PhasePhyto",
    "kaggle_json_drive_path": "/content/drive/MyDrive/kaggle.json",
    "download_plantvillage": True,
    "download_plantdoc": True,
    "download_plant_pathology_2021": True,
    "force_redownload": False,
    "recreate_archives": False,
    "rebuild_overlap": False,
    "allow_partial_overlap": False,
    "overlap_mode": "copy",  # use copy so the overlap archive is self-contained
    "hydrate_overlap_to_ssd": True,
    "keep_local_stage_data": False,
    "run_train": False,
    "run_eval_plantdoc": False,
    "run_eval_pp2021": False,
    "checkpoint_dir": "/content/drive/MyDrive/PhasePhyto/checkpoints/apple_overlap_plantdoc",
}

for k, v in CONFIG.items():
    print(f"{k}: {v}")


In [ ]:
# ============================================================
# MOUNT DRIVE + CLONE/INSTALL REPO + DEFINE PATHS
# ============================================================

from pathlib import Path
import os
import shutil
import subprocess
import sys
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

REPO_DIR = Path(CONFIG["repo_dir"])
if REPO_DIR.exists() and not (REPO_DIR / ".git").exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--branch", CONFIG["repo_branch"],
        CONFIG["repo_url"], str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", CONFIG["repo_branch"], "--quiet"], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet"], check=False)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DRIVE_PROJECT_DIR = Path(CONFIG["drive_project_dir"])
ARCHIVE_DIR = DRIVE_PROJECT_DIR / "data" / "archives"
OVERLAP_ARCHIVE = ARCHIVE_DIR / "apple_strict.tar"
LOCAL_STAGE_ROOT = Path("/content/phasephyto_stage")
LOCAL_RAW_PARENT = LOCAL_STAGE_ROOT / "raw"
LOCAL_BENCHMARK_PARENT = LOCAL_STAGE_ROOT / "benchmarks"
LOCAL_OVERLAP_BUILD_PARENT = LOCAL_STAGE_ROOT / "overlap"
LOCAL_OVERLAP_BUILD_ROOT = LOCAL_OVERLAP_BUILD_PARENT / "apple_strict"
LOCAL_DATA_ROOT = Path("/content/data")
LOCAL_OVERLAP_PARENT = LOCAL_DATA_ROOT / "overlap"
LOCAL_OVERLAP_ROOT = LOCAL_OVERLAP_PARENT / "apple_strict"

PLANTVILLAGE_ARCHIVE = ARCHIVE_DIR / "plantvillage.tar"
PLANTDOC_ARCHIVE = ARCHIVE_DIR / "plantdoc.tar"
PP2021_ARCHIVE = ARCHIVE_DIR / "plant_pathology_2021.tar"

LOCAL_PLANT_DISEASE_ROOT = LOCAL_RAW_PARENT / "plant_disease"
LOCAL_BENCHMARK_ROOT = LOCAL_BENCHMARK_PARENT / "plant_benchmarks"
PLANTVILLAGE_DIR = LOCAL_PLANT_DISEASE_ROOT / "plantvillage"
PLANTDOC_DIR = LOCAL_PLANT_DISEASE_ROOT / "plantdoc"
PP2021_DIR = LOCAL_BENCHMARK_ROOT / "plant_pathology_2021"

for path in [
    DRIVE_PROJECT_DIR,
    ARCHIVE_DIR,
    LOCAL_STAGE_ROOT,
    LOCAL_RAW_PARENT,
    LOCAL_BENCHMARK_PARENT,
    LOCAL_OVERLAP_BUILD_PARENT,
    LOCAL_DATA_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Repo ready:", REPO_DIR)
print("Archive dir:", ARCHIVE_DIR)
print("Local raw parent:", LOCAL_RAW_PARENT)
print("Local benchmark parent:", LOCAL_BENCHMARK_PARENT)
print("Local overlap build root:", LOCAL_OVERLAP_BUILD_ROOT)
print("Local hydrated overlap root:", LOCAL_OVERLAP_ROOT)


In [ ]:
# ============================================================
# HELPERS + KAGGLE CREDENTIALS + SELF-CONTAINED OVERLAP SUPPORT
# ============================================================

import csv
import json
import re
import tarfile
import tempfile

APPLE_STRICT_CLASSES = (
    "Apple___healthy",
    "Apple___Apple_scab",
    "Apple___Cedar_apple_rust",
)
APPLE_STRICT_SET = set(APPLE_STRICT_CLASSES)
PLANTDOC_APPLE_MAP = {
    "Apple leaf": "Apple___healthy",
    "Apple Scab Leaf": "Apple___Apple_scab",
    "Apple rust leaf": "Apple___Cedar_apple_rust",
}
PP2021_APPLE_MAP = {
    "healthy": "Apple___healthy",
    "scab": "Apple___Apple_scab",
    "rust": "Apple___Cedar_apple_rust",
}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
NOTEBOOK_HELPER_VERSION = "apple-overlap-colab-v3"


def run(cmd, *, capture: bool = False):
    print("\n$ " + " ".join(map(str, cmd)))
    result = subprocess.run(
        list(map(str, cmd)),
        check=False,
        text=True,
        capture_output=capture,
    )
    if capture and result.stdout:
        print(result.stdout)
    if capture and result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: "
            + " ".join(map(str, cmd))
        )
    return result


def ensure_kaggle_credentials() -> None:
    kaggle_json = Path.home() / ".kaggle" / "kaggle.json"
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    drive_copy = Path(CONFIG["kaggle_json_drive_path"])

    if kaggle_json.exists():
        os.chmod(kaggle_json, 0o600)
        print("Kaggle credentials already exist:", kaggle_json)
        return

    if drive_copy.exists():
        shutil.copy2(drive_copy, kaggle_json)
        os.chmod(kaggle_json, 0o600)
        print("Copied Kaggle credentials from Drive:", drive_copy)
        return

    print("Upload kaggle.json now (Kaggle > Account > Create New API Token).")
    from google.colab import files
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("kaggle.json was not uploaded.")
    shutil.move("kaggle.json", kaggle_json)
    shutil.copy2(kaggle_json, drive_copy)
    os.chmod(kaggle_json, 0o600)
    print("Saved reusable kaggle.json to:", drive_copy)


def create_or_refresh_tar(source_dir: Path, archive_path: Path, *, force: bool) -> None:
    if not source_dir.exists():
        raise FileNotFoundError(f"Source directory missing: {source_dir}")
    if archive_path.exists() and not force:
        print(f"Archive already exists: {archive_path}")
        return
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    if archive_path.exists():
        archive_path.unlink()
    with tarfile.open(archive_path, "w") as tf:
        tf.add(source_dir, arcname=source_dir.name)
    size_gb = archive_path.stat().st_size / 1e9
    print(f"Created archive: {archive_path} ({size_gb:.2f} GB)")


def extract_tar_to_parent(archive_path: Path, parent_dir: Path, *, clean_target: bool = True) -> Path:
    if not archive_path.exists():
        raise FileNotFoundError(f"Archive not found: {archive_path}")
    with tarfile.open(archive_path) as tf:
        top_levels = [member.name.split('/', 1)[0] for member in tf.getmembers() if member.name]
        top_level = next((name for name in top_levels if name and name != '.'), None)
        if top_level is None:
            raise RuntimeError(f"Could not determine top-level folder inside {archive_path}")
    target_dir = parent_dir / top_level
    if clean_target and target_dir.exists():
        shutil.rmtree(target_dir)
    parent_dir.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path) as tf:
        tf.extractall(parent_dir)
    print(f"Extracted {archive_path} -> {target_dir}")
    return target_dir


def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def count_image_files(folder: Path) -> int:
    if not folder.exists():
        return 0
    return sum(
        1
        for image_path in folder.iterdir()
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS
    )


def image_count(root: Path) -> int:
    if not root.exists():
        return 0
    return sum(
        1 for p in root.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )


def class_counts(root: Path) -> dict[str, int]:
    counts = {}
    if not root.exists():
        return counts
    for class_dir in sorted(p for p in root.iterdir() if p.is_dir()):
        n = count_image_files(class_dir)
        if n:
            counts[class_dir.name] = n
    return counts


def find_image_folder_candidate(root: Path) -> Path | None:
    candidates = []
    for p in root.rglob("*"):
        if not p.is_dir():
            continue
        counts = class_counts(p)
        if counts:
            candidates.append((len(counts), image_count(p), p))
    if not candidates:
        return None
    candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
    return candidates[0][2]


def copy_class_folders(src_root: Path, dest_root: Path) -> int:
    dest_root.mkdir(parents=True, exist_ok=True)
    copied = 0
    src_counts = class_counts(src_root)
    for class_dir in sorted(p for p in src_root.iterdir() if p.is_dir()):
        if src_counts.get(class_dir.name, 0) == 0:
            continue
        shutil.copytree(class_dir, dest_root / class_dir.name, dirs_exist_ok=True)
        copied += 1
    return copied


def resolve_image_folder(root: Path, preferred_splits=("test", "val", "valid", "validation", "train")) -> Path:
    root = Path(root)
    if class_counts(root):
        return root
    for split_name in preferred_splits:
        candidate = root / split_name
        if class_counts(candidate):
            return candidate
    return root


def download_plantvillage_local(output_dir: Path) -> None:
    dest = output_dir / "plantvillage"
    if dest.exists() and image_count(dest) > 0 and not CONFIG["force_redownload"]:
        print(f"PlantVillage already exists at {dest}, skipping download.")
        return
    if dest.exists():
        shutil.rmtree(dest)

    run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    tmp = Path(tempfile.mkdtemp(prefix="pv_dl_", dir=str(output_dir.parent)))
    try:
        run([
            "kaggle", "datasets", "download",
            "-d", "abdallahalidev/plantvillage-dataset",
            "-p", str(tmp),
        ])
        zip_files = list(tmp.glob("*.zip"))
        if not zip_files:
            raise RuntimeError(f"No PlantVillage zip downloaded to {tmp}")
        run(["unzip", "-q", str(zip_files[0]), "-d", str(tmp / "extracted")])
        color_candidates = [p for p in (tmp / "extracted").rglob("color") if p.is_dir()]
        src = color_candidates[0] if color_candidates else find_image_folder_candidate(tmp / "extracted")
        if src is None:
            raise RuntimeError("Could not find PlantVillage image-folder layout after extraction.")
        copied = copy_class_folders(src, dest)
        print(f"Copied {copied} PlantVillage class folders from {src}")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)


def download_plantdoc_local(output_dir: Path) -> None:
    dest = output_dir / "plantdoc"
    if dest.exists() and image_count(dest) > 0 and not CONFIG["force_redownload"]:
        print(f"PlantDoc already exists at {dest}, skipping download.")
        return
    if dest.exists():
        shutil.rmtree(dest)

    tmp_parent = Path(tempfile.mkdtemp(prefix="pd_dl_", dir=str(output_dir.parent)))
    tmp = tmp_parent / "repo"
    try:
        run([
            "git", "clone", "--depth", "1",
            "https://github.com/pratikkayal/PlantDoc-Dataset.git",
            str(tmp),
        ])
        dest.mkdir(parents=True, exist_ok=True)
        split_copied = False
        for split in ["train", "test"]:
            src_split = tmp / split
            if src_split.exists() and class_counts(src_split):
                copy_class_folders(src_split, dest / split)
                split_copied = True
        if not split_copied:
            src = find_image_folder_candidate(tmp)
            if src is None:
                raise RuntimeError("Could not find PlantDoc image-folder layout after clone.")
            copy_class_folders(src, dest)
    finally:
        shutil.rmtree(tmp_parent, ignore_errors=True)
    print(f"PlantDoc extracted to {dest} ({image_count(dest)} images)")


def normalize_pp2021_label(row: dict[str, str]) -> str:
    if row.get("labels"):
        text = row["labels"]
        parts = [
            normalize_name(part)
            for part in re.split(r"[,;+|]", text)
            if part.strip()
        ]
        return "__".join(parts) if parts else "unknown"
    if row.get("label"):
        return normalize_name(row["label"])
    active = []
    for key in ["complex", "frog_eye_leaf_spot", "healthy", "powdery_mildew", "rust", "scab"]:
        value = str(row.get(key, "")).strip().lower()
        if value in {"1", "1.0", "true", "yes"}:
            active.append(key)
    return "__".join(active) if active else "unknown"


def download_pp2021_local(output_dir: Path) -> None:
    dest = output_dir / "plant_pathology_2021"
    if dest.exists() and image_count(dest) > 0 and not CONFIG["force_redownload"]:
        print(f"Plant Pathology 2021 already exists at {dest}, skipping download.")
        return
    if dest.exists():
        shutil.rmtree(dest)

    run([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    tmp = Path(tempfile.mkdtemp(prefix="pp21_dl_", dir=str(output_dir.parent)))
    try:
        run([
            "kaggle", "competitions", "download",
            "-c", "plant-pathology-2021-fgvc8",
            "-p", str(tmp),
        ])
        zip_files = list(tmp.glob("*.zip"))
        if not zip_files:
            raise RuntimeError(f"No Plant Pathology 2021 zip downloaded to {tmp}")
        run(["unzip", "-q", str(zip_files[0]), "-d", str(tmp / "extracted")])
        extracted = tmp / "extracted"
        train_csv = next(iter(extracted.rglob("train.csv")), None)
        if train_csv is None:
            raise RuntimeError(f"Could not find train.csv under {extracted}")
        image_dirs = [
            p for p in extracted.rglob("*")
            if p.is_dir() and any(
                c.is_file() and c.suffix.lower() in IMAGE_EXTENSIONS
                for c in p.iterdir()
            )
        ]
        if not image_dirs:
            raise RuntimeError(f"Could not find extracted image directory under {extracted}")
        images_dir = max(image_dirs, key=lambda p: sum(1 for _ in p.iterdir()))
        dest.mkdir(parents=True, exist_ok=True)
        with open(train_csv, newline="") as f:
            reader = csv.DictReader(f)
            image_key = "image" if "image" in (reader.fieldnames or []) else "image_id"
            for row in reader:
                image_id = row.get(image_key)
                if not image_id:
                    continue
                class_name = normalize_pp2021_label(row)
                src_img = images_dir / image_id
                if not src_img.exists():
                    for ext in IMAGE_EXTENSIONS:
                        candidate = images_dir / f"{image_id}{ext}"
                        if candidate.exists():
                            src_img = candidate
                            break
                if not src_img.exists():
                    continue
                class_dir = dest / class_name
                class_dir.mkdir(parents=True, exist_ok=True)
                dst_name = src_img.name if src_img.suffix else f"{image_id}.jpg"
                shutil.copy2(src_img, class_dir / dst_name)
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
    print(f"Plant Pathology 2021 extracted to {dest} ({image_count(dest)} images)")


def inspect_apple_overlap_inline(
    plantvillage_root: Path,
    plantdoc_root: Path,
    pp2021_root: Path,
):
    pv_resolved = resolve_image_folder(plantvillage_root, ("train", "training", "val", "test"))
    pd_resolved = resolve_image_folder(plantdoc_root, ("test", "Test", "val", "valid"))
    pp_resolved = resolve_image_folder(pp2021_root, ("train", "val", "valid", "test"))

    pv_counts = {cls: count_image_files(pv_resolved / cls) for cls in APPLE_STRICT_CLASSES}
    pd_counts = {cls: 0 for cls in APPLE_STRICT_CLASSES}
    for raw_name, mapped_name in PLANTDOC_APPLE_MAP.items():
        pd_counts[mapped_name] += count_image_files(pd_resolved / raw_name)
    pp_counts = {cls: 0 for cls in APPLE_STRICT_CLASSES}
    for class_dir in sorted(p for p in pp_resolved.iterdir() if p.is_dir()):
        mapped = PP2021_APPLE_MAP.get(class_dir.name)
        if mapped is None:
            mapped = PP2021_APPLE_MAP.get(normalize_name(class_dir.name))
        if mapped is None and class_dir.name in APPLE_STRICT_SET:
            mapped = class_dir.name
        if mapped is None:
            continue
        pp_counts[mapped] += count_image_files(class_dir)

    datasets = {
        "plantvillage": pv_counts,
        "plantdoc": pd_counts,
        "plant_pathology_2021": pp_counts,
    }
    missing = {
        dataset_name: [cls for cls, count in counts.items() if count <= 0]
        for dataset_name, counts in datasets.items()
        if any(count <= 0 for count in counts.values())
    }
    return {
        "labels": list(APPLE_STRICT_CLASSES),
        "resolved_roots": {
            "plantvillage": str(pv_resolved),
            "plantdoc": str(pd_resolved),
            "plant_pathology_2021": str(pp_resolved),
        },
        "datasets": datasets,
        "missing_by_dataset": missing,
        "is_complete": not missing,
    }


def build_apple_overlap_inline(
    plantvillage_root: Path,
    plantdoc_root: Path,
    pp2021_root: Path,
    output_root: Path,
    *,
    mode: str,
    allow_missing: bool,
    clean: bool,
):
    report = inspect_apple_overlap_inline(plantvillage_root, plantdoc_root, pp2021_root)
    if clean and output_root.exists():
        shutil.rmtree(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    if not allow_missing and not report["is_complete"]:
        report_path = output_root / "overlap_manifest.json"
        report_path.write_text(json.dumps(report, indent=2) + "\n")
        raise RuntimeError(
            "Strict apple overlap is incomplete. Missing classes by dataset: "
            + json.dumps(report["missing_by_dataset"], indent=2)
        )

    def materialize(src: Path, dst: Path):
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.exists():
            return
        if mode == "symlink":
            try:
                dst.symlink_to(src.resolve())
                return
            except OSError:
                pass
        shutil.copy2(src, dst)

    pv_resolved = Path(report["resolved_roots"]["plantvillage"])
    pd_resolved = Path(report["resolved_roots"]["plantdoc"])
    pp_resolved = Path(report["resolved_roots"]["plant_pathology_2021"])

    for class_name in APPLE_STRICT_CLASSES:
        src_dir = pv_resolved / class_name
        if not src_dir.exists():
            continue
        for image_path in sorted(src_dir.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                materialize(image_path, output_root / "plantvillage" / class_name / image_path.name)

    for raw_name, mapped_name in PLANTDOC_APPLE_MAP.items():
        src_dir = pd_resolved / raw_name
        if not src_dir.exists():
            continue
        for image_path in sorted(src_dir.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                filename = f"{raw_name.replace(' ', '_')}__{image_path.name}"
                materialize(image_path, output_root / "plantdoc" / mapped_name / filename)

    for class_dir in sorted(p for p in pp_resolved.iterdir() if p.is_dir()):
        mapped = PP2021_APPLE_MAP.get(class_dir.name)
        if mapped is None:
            mapped = PP2021_APPLE_MAP.get(normalize_name(class_dir.name))
        if mapped is None and class_dir.name in APPLE_STRICT_SET:
            mapped = class_dir.name
        if mapped is None:
            continue
        for image_path in sorted(class_dir.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                filename = f"{class_dir.name}__{image_path.name}"
                materialize(image_path, output_root / "plant_pathology_2021" / mapped / filename)

    final_report = inspect_apple_overlap_inline(
        output_root / "plantvillage",
        output_root / "plantdoc",
        output_root / "plant_pathology_2021",
    )
    final_report["output_root"] = str(output_root)
    final_report["mode"] = mode
    (output_root / "overlap_manifest.json").write_text(json.dumps(final_report, indent=2) + "\n")
    return final_report


def ensure_overlap_configs() -> tuple[Path, Path]:
    cfg_dir = REPO_DIR / "configs"
    cfg_dir.mkdir(parents=True, exist_ok=True)
    plantdoc_cfg = cfg_dir / "apple_overlap_plantdoc.yaml"
    pp2021_cfg = cfg_dir / "apple_overlap_pp2021.yaml"

    plantdoc_cfg.write_text(
        "# Strict apple-overlap benchmark: PlantVillage source -> PlantDoc target\n"
        "seed: 42\n"
        "device: \"cuda\"\n"
        "checkpoint_dir: \"checkpoints/apple_overlap_plantdoc\"\n"
        "project_name: \"phasephyto-apple-overlap-plantdoc\"\n\n"
        "model:\n"
        "  backbone_name: \"vit_base_patch16_224\"\n"
        "  fusion_dim: 256\n"
        "  pretrained_backbone: true\n"
        "  freeze_backbone: false\n\n"
        "training:\n"
        "  lr: 5.0e-5\n"
        "  weight_decay: 1.0e-2\n"
        "  epochs: 30\n"
        "  warmup_epochs: 3\n"
        "  batch_size: 32\n"
        "  patience: 8\n"
        "  loss: \"focal\"\n"
        "  focal_gamma: 2.0\n\n"
        "data:\n"
        "  root: \"data/overlap/apple_strict\"\n"
        "  use_case: \"plant_disease\"\n"
        "  image_size: 224\n"
        "  val_split: 0.2\n"
        "  source_dir: \"data/overlap/apple_strict/plantvillage\"\n"
        "  target_dir: \"data/overlap/apple_strict/plantdoc\"\n"
        "  train_dir: \"\"\n"
        "  val_dir: \"\"\n"
        "  eval_source_dir: \"data/overlap/apple_strict/plantvillage\"\n"
        "  eval_target_dir: \"data/overlap/apple_strict/plantdoc\"\n"
    )
    pp2021_cfg.write_text(
        "# Strict apple-overlap benchmark: PlantVillage source -> Plant Pathology 2021 target\n"
        "seed: 42\n"
        "device: \"cuda\"\n"
        "checkpoint_dir: \"checkpoints/apple_overlap_pp2021\"\n"
        "project_name: \"phasephyto-apple-overlap-pp2021\"\n\n"
        "model:\n"
        "  backbone_name: \"vit_base_patch16_224\"\n"
        "  fusion_dim: 256\n"
        "  pretrained_backbone: true\n"
        "  freeze_backbone: false\n\n"
        "training:\n"
        "  lr: 5.0e-5\n"
        "  weight_decay: 1.0e-2\n"
        "  epochs: 30\n"
        "  warmup_epochs: 3\n"
        "  batch_size: 32\n"
        "  patience: 8\n"
        "  loss: \"focal\"\n"
        "  focal_gamma: 2.0\n\n"
        "data:\n"
        "  root: \"data/overlap/apple_strict\"\n"
        "  use_case: \"plant_disease\"\n"
        "  image_size: 224\n"
        "  val_split: 0.2\n"
        "  source_dir: \"data/overlap/apple_strict/plantvillage\"\n"
        "  target_dir: \"data/overlap/apple_strict/plant_pathology_2021\"\n"
        "  train_dir: \"\"\n"
        "  val_dir: \"\"\n"
        "  eval_source_dir: \"data/overlap/apple_strict/plantvillage\"\n"
        "  eval_target_dir: \"data/overlap/apple_strict/plant_pathology_2021\"\n"
    )
    return plantdoc_cfg, pp2021_cfg


ensure_kaggle_credentials()


## Preflight: what will be reused vs rebuilt

This cell checks which tar archives already exist on Drive and tells you whether the notebook will reuse them or rebuild/download them based on the current config flags.


In [ ]:
# ============================================================
# PREFLIGHT: DRIVE TAR STATUS + PLANNED ACTIONS
# ============================================================

preflight_rows = [
    ("plantvillage", PLANTVILLAGE_ARCHIVE, CONFIG["download_plantvillage"]),
    ("plantdoc", PLANTDOC_ARCHIVE, CONFIG["download_plantdoc"]),
    ("plant_pathology_2021", PP2021_ARCHIVE, CONFIG["download_plant_pathology_2021"]),
    ("apple_overlap", OVERLAP_ARCHIVE, True),
]

print("Preflight status:\n")
for name, archive_path, enabled in preflight_rows:
    exists = archive_path.exists()
    status = "present" if exists else "missing"

    if name == "apple_overlap":
        if exists and not CONFIG["rebuild_overlap"] and not CONFIG["recreate_archives"]:
            action = "reuse overlap tar from Drive"
        elif CONFIG["rebuild_overlap"] or CONFIG["recreate_archives"]:
            action = "rebuild overlap + refresh tar"
        else:
            action = "build overlap and save tar"
    else:
        if not enabled:
            action = "skip (disabled in CONFIG)"
        elif exists and not CONFIG["force_redownload"] and not CONFIG["recreate_archives"]:
            action = "reuse tar from Drive and unpack to SSD"
        else:
            action = "download/prep on SSD and save tar to Drive"

    print(f"- {name}: {status} | {archive_path}")
    print(f"  planned action: {action}")

print("\nFlag summary:")
print(f"  force_redownload={CONFIG['force_redownload']}")
print(f"  recreate_archives={CONFIG['recreate_archives']}")
print(f"  rebuild_overlap={CONFIG['rebuild_overlap']}")
print(f"  allow_partial_overlap={CONFIG['allow_partial_overlap']}")
print(f"  hydrate_overlap_to_ssd={CONFIG['hydrate_overlap_to_ssd']}")


In [ ]:
# ============================================================
# DOWNLOAD RAW DATASETS TO LOCAL SSD, THEN SAVE TAR ARCHIVES TO DRIVE
# ============================================================

dataset_jobs = [
    {
        "enabled": CONFIG["download_plantvillage"],
        "name": "plantvillage",
        "archive": PLANTVILLAGE_ARCHIVE,
        "expected_dir": PLANTVILLAGE_DIR,
        "download_fn": download_plantvillage_local,
    },
    {
        "enabled": CONFIG["download_plantdoc"],
        "name": "plantdoc",
        "archive": PLANTDOC_ARCHIVE,
        "expected_dir": PLANTDOC_DIR,
        "download_fn": download_plantdoc_local,
    },
    {
        "enabled": CONFIG["download_plant_pathology_2021"],
        "name": "plant_pathology_2021",
        "archive": PP2021_ARCHIVE,
        "expected_dir": PP2021_DIR,
        "download_fn": download_pp2021_local,
    },
]

for job in dataset_jobs:
    if not job["enabled"]:
        print(f"Skipping {job['name']} (disabled in CONFIG).")
        continue

    archive_path = job["archive"]
    expected_dir = job["expected_dir"]

    if archive_path.exists() and not CONFIG["force_redownload"] and not CONFIG["recreate_archives"]:
        print(f"Using existing archive for {job['name']}: {archive_path}")
        extract_tar_to_parent(archive_path, expected_dir.parent, clean_target=True)
        continue

    if expected_dir.exists():
        shutil.rmtree(expected_dir)
    expected_dir.parent.mkdir(parents=True, exist_ok=True)
    job["download_fn"](expected_dir.parent)
    create_or_refresh_tar(expected_dir, archive_path, force=True)


In [ ]:
# ============================================================
# ARCHIVE STATUS ON DRIVE
# ============================================================

print("Raw archive status:")
for archive_path in [PLANTVILLAGE_ARCHIVE, PLANTDOC_ARCHIVE, PP2021_ARCHIVE]:
    status = "present" if archive_path.exists() else "missing"
    print(f"  {archive_path.name}: {status}")
overlap_status = "present" if OVERLAP_ARCHIVE.exists() else "missing"
print(f"Overlap archive status: {OVERLAP_ARCHIVE.name}: {overlap_status}")
print(
    "If a later session loses local SSD data, rerun the archive cells to "
    "unpack existing Drive tar files instead of re-downloading."
)


## Overlap coverage precheck

This cell inspects the apple-overlap coverage across PlantVillage, PlantDoc, and Plant Pathology 2021 before attempting to build the overlap subset. If a class is missing, you can either fix the source dataset or set `allow_partial_overlap=True` to continue with a partial subset.


In [ ]:
if globals().get("NOTEBOOK_HELPER_VERSION") != "apple-overlap-colab-v3":
    raise RuntimeError(
        "Notebook helpers are stale in memory. Rerun the HELPERS cell (cell 3) "
        "or restart the runtime and Run all before continuing."
    )

# ============================================================
# PREFLIGHT: APPLE OVERLAP COVERAGE REPORT
# ============================================================

overlap_report = inspect_apple_overlap_inline(
    plantvillage_root=PLANTVILLAGE_DIR,
    plantdoc_root=PLANTDOC_DIR,
    pp2021_root=PP2021_DIR,
)
if not isinstance(overlap_report, dict):
    raise RuntimeError(
        "inspect_apple_overlap_inline returned an invalid value. Rerun the HELPERS cell "
        "or restart the runtime and Run all."
    )
print(json.dumps(overlap_report, indent=2))
if not CONFIG["allow_partial_overlap"] and not overlap_report["is_complete"]:
    raise RuntimeError(
        "Strict apple overlap is incomplete. Set CONFIG['allow_partial_overlap']=True "
        "to continue with a partial subset, or fix the missing classes shown above."
    )


In [ ]:
if globals().get("NOTEBOOK_HELPER_VERSION") != "apple-overlap-colab-v3":
    raise RuntimeError(
        "Notebook helpers are stale in memory. Rerun the HELPERS cell (cell 3) "
        "or restart the runtime and Run all before continuing."
    )

# ============================================================
# BUILD STRICT APPLE OVERLAP ONLY IF NEEDED, THEN SAVE TAR TO DRIVE
# ============================================================

if OVERLAP_ARCHIVE.exists() and not CONFIG["rebuild_overlap"] and not CONFIG["recreate_archives"]:
    print(f"Using existing overlap archive from Drive: {OVERLAP_ARCHIVE}")
else:
    if CONFIG["rebuild_overlap"] and LOCAL_OVERLAP_BUILD_ROOT.exists():
        shutil.rmtree(LOCAL_OVERLAP_BUILD_ROOT)

    overlap_report = build_apple_overlap_inline(
        plantvillage_root=PLANTVILLAGE_DIR,
        plantdoc_root=PLANTDOC_DIR,
        pp2021_root=PP2021_DIR,
        output_root=LOCAL_OVERLAP_BUILD_ROOT,
        mode=CONFIG["overlap_mode"],
        allow_missing=CONFIG["allow_partial_overlap"],
        clean=True,
    )
    print(json.dumps(overlap_report, indent=2))
    create_or_refresh_tar(
        LOCAL_OVERLAP_BUILD_ROOT,
        OVERLAP_ARCHIVE,
        force=CONFIG["recreate_archives"] or CONFIG["rebuild_overlap"],
    )


In [ ]:
# ============================================================
# HYDRATE OVERLAP TAR TO COLAB SSD (/content/data) FOR TRAINING SPEED
# ============================================================

if CONFIG["hydrate_overlap_to_ssd"]:
    if LOCAL_OVERLAP_PARENT.exists():
        shutil.rmtree(LOCAL_OVERLAP_PARENT)
    extract_tar_to_parent(OVERLAP_ARCHIVE, LOCAL_OVERLAP_PARENT, clean_target=False)
    print("Hydrated overlap archive to SSD:", LOCAL_OVERLAP_ROOT)
else:
    print("Skipped overlap hydration; use the overlap archive directly later if you want.")

if not CONFIG["keep_local_stage_data"]:
    if LOCAL_RAW_PARENT.exists():
        shutil.rmtree(LOCAL_RAW_PARENT)
    if LOCAL_BENCHMARK_PARENT.exists():
        shutil.rmtree(LOCAL_BENCHMARK_PARENT)
    if LOCAL_OVERLAP_BUILD_ROOT.exists():
        shutil.rmtree(LOCAL_OVERLAP_BUILD_ROOT)
    print("Cleaned local stage folders after archiving.")
else:
    print("Keeping local stage folders on SSD because CONFIG['keep_local_stage_data'] = True.")


In [ ]:
# ============================================================
# SANITY CHECK COUNTS
# ============================================================

for dataset_name in ["plantvillage", "plantdoc", "plant_pathology_2021"]:
    ds_root = LOCAL_OVERLAP_ROOT / dataset_name
    print(f"\nDATASET: {dataset_name}")
    for class_name in APPLE_STRICT_CLASSES:
        class_dir = ds_root / class_name
        n = len([p for p in class_dir.glob("*") if p.is_file()]) if class_dir.exists() else 0
        print(f"  {class_name}: {n}")


In [ ]:
if globals().get("NOTEBOOK_HELPER_VERSION") != "apple-overlap-colab-v3":
    raise RuntimeError(
        "Notebook helpers are stale in memory. Rerun the HELPERS cell (cell 3) "
        "or restart the runtime and Run all before continuing."
    )

# ============================================================
# TRAIN: PlantVillage source -> PlantDoc target overlap benchmark
# ============================================================

PLANTDOC_CFG, PP2021_CFG = ensure_overlap_configs()

if CONFIG["run_train"]:
    run([
        sys.executable, "-m", "phasephyto.train",
        "--config", str(PLANTDOC_CFG),
        "--override",
        f"checkpoint_dir={CONFIG['checkpoint_dir']}",
        f"data.root={LOCAL_OVERLAP_ROOT}",
        f"data.source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
        f"data.eval_source_dir={LOCAL_OVERLAP_ROOT / 'plantvillage'}",
        f"data.eval_target_dir={LOCAL_OVERLAP_ROOT / 'plantdoc'}",
    ], capture=True)
else:
    print("Set CONFIG['run_train'] = True to launch training from this notebook.")


In [ ]:
if globals().get("NOTEBOOK_HELPER_VERSION") != "apple-overlap-colab-v3":
    raise RuntimeError(
        "Notebook helpers are stale in memory. Rerun the HELPERS cell (cell 3) "
        "or restart the runtime and Run all before continuing."
    )

# ============================================================
# EVAL 1: PlantVillage -> PlantDoc overlap
# ============================================================

PLANTDOC_CFG, PP2021_CFG = ensure_overlap_configs()

if CONFIG["run_eval_plantdoc"]:
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", str(PLANTDOC_CFG),
        "--checkpoint", str(Path(CONFIG["checkpoint_dir"]) / "best_model.pt"),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plantdoc"),
        "--output", str(Path(CONFIG["checkpoint_dir"]) / "eval_plantdoc.json"),
    ], capture=True)
else:
    print("Set CONFIG['run_eval_plantdoc'] = True to run PlantDoc evaluation.")


In [ ]:
if globals().get("NOTEBOOK_HELPER_VERSION") != "apple-overlap-colab-v3":
    raise RuntimeError(
        "Notebook helpers are stale in memory. Rerun the HELPERS cell (cell 3) "
        "or restart the runtime and Run all before continuing."
    )

# ============================================================
# EVAL 2: PlantVillage -> Plant Pathology 2021 overlap
# ============================================================

PLANTDOC_CFG, PP2021_CFG = ensure_overlap_configs()

if CONFIG["run_eval_pp2021"]:
    run([
        sys.executable, "-m", "phasephyto.evaluate",
        "--config", str(PP2021_CFG),
        "--checkpoint", str(Path(CONFIG["checkpoint_dir"]) / "best_model.pt"),
        "--source-dir", str(LOCAL_OVERLAP_ROOT / "plantvillage"),
        "--target-dir", str(LOCAL_OVERLAP_ROOT / "plant_pathology_2021"),
        "--output", str(Path(CONFIG["checkpoint_dir"]) / "eval_pp2021.json"),
    ], capture=True)
else:
    print("Set CONFIG['run_eval_pp2021'] = True to run Plant Pathology 2021 evaluation.")


## Combined evaluation summary

This cell reads `eval_plantdoc.json` and `eval_pp2021.json` if present, writes a combined JSON + CSV summary to Drive, and saves a simple comparison bar chart.


In [ ]:
if globals().get("NOTEBOOK_HELPER_VERSION") != "apple-overlap-colab-v3":
    raise RuntimeError(
        "Notebook helpers are stale in memory. Rerun the HELPERS cell (cell 3) "
        "or restart the runtime and Run all before continuing."
    )

# ============================================================
# COMBINED EVALUATION SUMMARY (saved to Drive)
# ============================================================

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

checkpoint_dir = Path(CONFIG["checkpoint_dir"])
plantdoc_eval = checkpoint_dir / "eval_plantdoc.json"
pp2021_eval = checkpoint_dir / "eval_pp2021.json"

rows = []
for target_name, eval_path in [
    ("plantdoc", plantdoc_eval),
    ("plant_pathology_2021", pp2021_eval),
]:
    if not eval_path.exists():
        print(f"Missing eval file for {target_name}: {eval_path}")
        continue
    payload = json.loads(eval_path.read_text())
    rows.append({
        "target": target_name,
        "source_accuracy": payload.get("source", {}).get("accuracy"),
        "target_accuracy": payload.get("target", {}).get("accuracy"),
        "accuracy_drop": payload.get("delta", {}).get("accuracy_drop"),
        "source_f1_macro": payload.get("source", {}).get("f1_macro"),
        "target_f1_macro": payload.get("target", {}).get("f1_macro"),
        "f1_drop": payload.get("delta", {}).get("f1_drop"),
        "use_case": payload.get("use_case"),
        "eval_json": str(eval_path),
    })

if not rows:
    raise RuntimeError(
        "No evaluation JSON files found. Run at least one eval cell first."
    )

summary_df = pd.DataFrame(rows).sort_values("target").reset_index(drop=True)
summary_json = checkpoint_dir / "apple_overlap_eval_summary.json"
summary_csv = checkpoint_dir / "apple_overlap_eval_summary.csv"
summary_md_path = checkpoint_dir / "apple_overlap_eval_summary.md"
summary_plot = checkpoint_dir / "apple_overlap_eval_summary.png"

summary_json.write_text(json.dumps(rows, indent=2) + "\n")
summary_df.to_csv(summary_csv, index=False)

bar_df = summary_df.set_index("target")[["target_accuracy", "target_f1_macro"]]
ax = bar_df.plot(kind="bar", figsize=(8, 4), ylim=(0, 1), rot=0)
ax.set_title("Apple overlap target performance")
ax.set_ylabel("score")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(summary_plot, dpi=140, bbox_inches="tight")
plt.show()

summary_lines = [
    "# Apple Overlap Evaluation Summary",
    "",
    f"- Checkpoint dir: `{checkpoint_dir}`",
    f"- PlantDoc eval: {'present' if plantdoc_eval.exists() else 'missing'}`{plantdoc_eval}`",
    f"- Plant Pathology 2021 eval: {'present' if pp2021_eval.exists() else 'missing'}`{pp2021_eval}`",
    "",
    "| Target | Target acc | Target F1 | Acc drop | F1 drop |",
    "|---|---:|---:|---:|---:|",
]
for row in rows:
    summary_lines.append(
        f"| {row['target']} | {row['target_accuracy']:.4f} | {row['target_f1_macro']:.4f} | {row['accuracy_drop']:+.4f} | {row['f1_drop']:+.4f} |"
    )
summary_md_path.write_text("\n".join(summary_lines) + "\n")

print("Saved combined overlap summary files:")
print("  JSON:", summary_json)
print("  CSV: ", summary_csv)
print("  MD:  ", summary_md_path)
print("  PNG: ", summary_plot)
summary_df


## Optional next step: batch inference notebook

After you have the four ablation checkpoints, open:

- `notebooks/PhasePhyto_Batch_Inference.ipynb`

and configure `DATASET_RUNS` with:

- `/content/data/overlap/apple_strict/plantdoc`
- `/content/data/overlap/apple_strict/plant_pathology_2021`

using `/content/data/overlap/apple_strict/plantvillage` as `class_to_idx_source`.
